# **PROYEK UAS**

## Nama Proyek : **Melakukan klasifikasi time series pada dataset DucksAndGeese**

### A. **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*)

**1. *Business Understanding* (Pemahaman Bisnis)**

**1.1 Latar Belakang**

Identifikasi spesies burung merupakan bagian penting dalam bidang ekologi dan konservasi, khususnya untuk keperluan monitoring populasi dan pengamatan biodiversitas. Namun, identifikasi burung secara visual sering kali sulit dilakukan karena keterbatasan kondisi lapangan, seperti burung yang tersembunyi di vegetasi, aktif pada waktu tertentu, atau berada di lokasi yang sulit dijangkau oleh pengamat.

Salah satu pendekatan alternatif yang banyak digunakan adalah bioakustik, yaitu identifikasi spesies berdasarkan karakteristik suara. Dataset **DucksAndGeese** yang berasal dari *UEA Time Series Archive* menyediakan sinyal audio berbentuk *time series* yang merepresentasikan lima spesies burung air, yang terdiri dari dua spesies bebek (*duck*) dan tiga spesies angsa (*geese*).

Permasalahan utama pada dataset ini adalah bagaimana mengklasifikasikan sinyal *audio time series* tersebut ke dalam kategori spesies burung yang benar, mengingat adanya variasi pola suara, *noise* lingkungan, serta kemiripan karakteristik frekuensi antar spesies.

**1.2 Tujuan Proyek**

Membangun dan mengevaluasi model *time series classification* berbasis *machine learning* yang mampu mengklasifikasikan sinyal audio burung ke dalam lima kelas spesies berikut:
- Black-bellied Whistling Duck
- Canadian Goose
- Greylag Goose
- Pink-footed Goose
- White-faced Whistling Duck
Permasalahan ini termasuk dalam klasifikasi multikelas (*multiclass classification*) dengan lima label kelas diskrit, di mana setiap sinyal audio direpresentasikan sebagai satu *instance data*.

**1.3 Manfaat**

Hasil dari klasifikasi sinyal audio pada penelitian ini diharapkan dapat dimanfaatkan sebagai dasar pengembangan **sistem identifikasi bioakustik otomatis**, dengan manfaat sebagai berikut:
- Mendukung monitoring fauna secara otomatis tanpa ketergantungan pada observasi visual.
- Membantu pengamatan dan pemetaan biodiversitas burung di alam liar.
- Mengurangi ketergantungan pada identifikasi manual oleh manusia, yang cenderung memerlukan waktu, tenaga, dan keahlian khusus.
- Menjadi dasar pengembangan aplikasi berbasis audio untuk keperluan ekologi dan konservasi satwa liar.

**1.4 Kriteria kesuksesan**

Model dikatakan berhasil jika mencapai:
- **Accuracy ≥ 85%** untuk *baseline*.
- **F1-score** per kelas **≥ 80%**, untuk memastikan performa yang seimbang pada seluruh spesies.
- Aplikasi Streamlit berjalan lancar dan mampu menerima input serta menampilkan prediksi dan *confidence score*.

**2. *Data Understanding* (Pemahaman Data)**

**2.1 Gambaran Umum Dataset**

Dataset **DucksAndGeese** berasal dari *UEA Time Series Archive* dan berisi sinyal *time series audio univariate* yang merepresentasikan suara dari 5 spesies burung air, terdiri atas 2 jenis bebek (*duck*) dan 3 jenis angsa (*geese*).

Dataset ini digunakan untuk permasalahan klasifikasi time series multikelas, di mana setiap *instance* berupa satu rekaman audio mono yang telah diproses menjadi panjang sinyal yang seragam. Dataset ini merupakan versi *univariate* dari dataset **DuckDuckGeese** yang bersifat *multivariate*.

Setiap *instance* direpresentasikan sebagai urutan nilai amplitudo audio (*raw waveform*) yang kemudian dipasangkan dengan label kelas spesies.

Berdasarkan dokumentasi *UEA Time Series Archive* dan file **.ts**, spesifikasi dataset adalah sebagai berikut:

Tabel Spesifikasi Dataset
| Properti                             | Nilai                                  |
|--------------------------------------|----------------------------------------|
| *Train Size*                         | 50                                     |
| *Test Size*                          | 50                                     |
| *Total Instances*                    | 100                                    |
| *Time Series Length*                 | 236,784 titik data per *instance*      |
| *Number of Classes*                  | 5                                      |
| *Dimensions*                         | 1 (*audio waveform univariate*)        |
| *Datatype*                           | Float (*amplitudo audio*)              |
| *Data format*                        | AUDIO (*raw waveform*)                 |
| *Sample rate* setelah *preprocessing*| 44,100 Hz                              |
| *Data Split*                         | TRAIN dan TEST                         |
| Sumber                               | www.xenocanto.com                      |
Dengan asumsi *sample rate* 44.100 Hz, setiap rekaman memiliki durasi sekitar:
236.784 / 44.100 ≈ 5,37 detik

Tabel Kelas dan Distribusi Label
| Label | Spesies                          | Jumlah |
|-------|----------------------------------|--------|
| 0     | Black-bellied Whistling Duck     | 20     |
| 1     | Canadian Goose                   | 20     |
| 2     | Greylag Goose                    | 20     |
| 3     | Pink-footed Goose                | 20     |
| 4     | White-faced Whistling Duck       | 20     |
Karena jumlah data tiap kelas sama, dataset tidak mengalami *class imbalance*, sehingga tidak diperlukan teknik *oversampling* atau *undersampling* pada tahap *preprocessing*.

**2.2 Asal-usul dan Proses Pembentukan Data**

Berdasarkan dokumentasi **UEA Archive**:
- Audio berasal dari rekaman lapangan pada platform **Xeno-Canto**, sebuah repositori suara burung dan komunitas *ornithology*.
- Rekaman memiliki variasi *sample rate* dan durasi.
- Seluruh audio kemudian:
    1. Diseragamkan *sample rate*-nya yang awalnya berbeda-beda (misalnya 48kHz, 96kHz, 22kHz) yaitu *downsampling* ke 44.100 Hz.
    2. Dipangkas (*truncation*) ke panjang 236.784 data point, yang merupakan panjang terpendek dari seluruh rekaman karena panjang aslinya bervariasi.
- Rekaman suara berasal dari kategori kualitas A (noise rendah) atau B (noise sedang), berarti suara cukup bersih.
- Dataset telah disediakan dalam bentuk **TRAIN** (50) dan **TEST** (50), sehingga evaluasi dapat dilakukan langsung pada *test set* tanpa kebocoran data. Artinya, setiap *instance* merepresentasikan potongan audio berdurasi sama, sehingga model fokus pada pola temporal suara, bukan perbedaan panjang sinyal. Untuk pengembangan, split ulang TRAIN menjadi *train* / *validation* (80/20) untuk *tuning hyperparameter* dan memilih model sebelum uji pada TEST final.

**2.3 Struktur Data di Dalam File .ts**

Setelah membuka file TRAIN dan TEST:
- Setiap baris merepresentasikan 1 *instance*
- Format: TS format (UEA standard)
- Setiap baris berisi:
    v1, v2, v3, ..., v236784 : classLabel
- classLabel menunjukkan kelas spesies (0–4)
- Nilai v merupakan nilai amplitudo audio bertipe float
    ex : 0.0012, -0.0031, 0.0042, ... , -0.0008 : 3 (Artinya data tersebut adalah Pink-footed Goose)

**2.4 Karakteristik Data**

**2.4.1 *Univariate Long Sequence***

- Panjang data sangat besar (236k titik)
- Membutuhkan:
    - optimasi memori
    - kemungkinan segmentasi / ekstraksi fitur
- Pendekatan:
    - CNN-*based Time Series Classification*
    - *Distance-based method* (DTW)
    - *Feature-based ML* (untuk ekstraksi)

**2.4.2 *Balanced Classes***

Semua kelas masing-masing 20 sampel, sehingga:
- Tidak perlu balancing
- Setiap kelas memiliki jumlah instance yang sama
- Evaluasi model lebih stabil
- Accuracy dan Macro-F1 relevan digunakan

**2.4.3 *High Variability***

Karena suara burung:
- Rekaman berasal dari lingkungan alami, sehingga :
    - Banyak noise lingkungan
    - Bentuk gelombang sangat bervariasi
    - *High-frequency chirp*/*whistle patterns*

**2.4.4 *Data Challenges***

- Ukuran input besar (*memory heavy*)
- Harus distandarisasi (z-score)
- Perlu *filtering* (*bandpass* 300–8000 Hz)
- Potensial *cropping* / *feature extraction* (MFCC)

**2.5 EDA (*Exploratory Data Analysis*)**

**2.5.1 Struktur Dataset**

| File                   | Isi                              |
|------------------------|----------------------------------|
| DucksAndGeese_TRAIN.ts | 50 instance, univariate, labeled |
| DucksAndGeese_TEST.ts  | 50 instance, univariate, labeled |

**2.5.2 Basic Stats**

| Statistik              | Nilai                      |
|------------------------|----------------------------|
| Total sample           | 100                        |
| Panjang time series    | 236,784 *point per sample* |
| Channel                | 1 (mono)                   |
| Durasi estimasi        | 5.37 detik per sample      |

**2.5.3 Distribusi Label**

Semua kelas memiliki 20 sampel ***balanced***.
| Label | Nama Spesies                      | Jumlah |
|-------|-----------------------------------|--------|
| 0     | *Black-bellied Whistling Duck*    | 20     |
| 1     | *Canadian Goose*                  | 20     |
| 2     | *Greylag Goose*                   | 20     |
| 3     | *Pink-footed Goose*               | 20     |
| 4     | *White-faced Whistling Duck*      | 20     |

**2.5.4 *Visualisasi Waveform***

Tujuan EDA pada *audio time series*:
- melihat noise
- melihat pola amplitudo
- melihat perbedaan antar spesies

Insight yang diharapkan:
1. *Canadian Goose* = nada lebih rendah (gelombang besar & lambat)
2. *Whistling Duck* = suara nyaring (gelombang cepat & rapat)
3. *Greylag Goose* = tone cenderung clustering

**2.5.5 Analisis Statistik per Kelas**

Hitung:
- mean amplitude
- standard deviation
- zero crossing count
- RMS energy
ex :    | Kelas | Mean | Std  | ZeroCrossRate   | RMS |
        |-------|------|------|---------------- |-----|
        | 0     | …    | …    | …               | …   |

**3. *Data Preparation* / Preprocessing**

Tahap Data Preparation bertujuan untuk mengubah sinyal audio mentah (*raw waveform*) menjadi representasi fitur yang lebih ringkas, informatif, dan efisien untuk proses pemodelan *machine learning*. Karena data berupa *time series audio* berdimensi sangat besar, *preprocessing* menjadi tahap krusial dalam *pipeline* penelitian ini.

**3.1 Normalisasi Sinyal**

Setiap sinyal audio dinormalisasi menggunakan *z-score normalization*:
    X_norm = (X - μ​) / σ
dengan:
- μ = nilai rata-rata sinyal
- σ = standar deviasi sinyal

Gunanya:
- mengurangi perbedaan amplitude antar *recording*
- Menghindari dominasi nilai ekstrem
- Mempercepat konvergensi model
Normalisasi dilakukan per *instance*, sehingga setiap rekaman memiliki skala amplitudo yang sebanding.

**3.2 *Downsampling***

Dataset telah memiliki *sample rate* 44.100 Hz, namun pada penelitian ini *downsampling* bersifat opsional dan hanya dilakukan apabila diperlukan untuk efisiensi komputasi, mengingat dataset **DucksAndGeese** telah melalui **preprocessing** dasar berupa penyeragaman *sample rate* menjadi 44.100 Hz oleh peneliti sebelumnya.

Manfaat:
- Mengurangi ukuran data secara signifikan
- Mempercepat proses ekstraksi fitur
- Suara burung umumnya berada pada rentang frekuensi < 10 kHz, sehingga informasi utama tetap terjaga
*Downsampling* hanya dilakukan apabila dibutuhkan untuk efisiensi komputasi dan tidak mengubah label atau struktur kelas.

**3.3 *Feature Extraction***

Karena penggunaan waveform mentah tidak efisien untuk algoritma machine learning klasik, dilakukan ekstraksi fitur audio yang merepresentasikan karakteristik temporal dan spektral sinyal. Ekstraksi fitur dari audio:

1. MFCC (*Mel-frequency Cepstral Coefficients*)
MFCC digunakan sebagai fitur utama karena efektif merepresentasikan karakteristik suara.
- Jumlah koefisien: 13
- Mewakili pola spektral yang relevan dengan persepsi suara

2. *Zero Crossing Rate* (ZCR)
ZCR mengukur jumlah transisi sinyal melewati nol.
- Menggambarkan karakter frekuensi suara
- Suara whistling cenderung memiliki ZCR lebih tinggi

3. *Spectral Centroid*
Menunjukkan pusat massa spektrum frekuensi.
- menentukan “kecerahan” suara

4. *RMS Energy*
Mengukur energi rata-rata sinyal.
- melihat power sinyal
Hasil akhirnya berupa tabel fitur seperti:
| mfcc1  | mfcc2  | …  | ZCR | centroid | RMS  | label  |
|--------|--------|----|-----|----------|------|--------|

**3.4 Transformasi ke Domain Frekuensi**

Untuk sinyal audio burung, representasi frekuensi lebih deskriptif dibanding waveform mentah.
Metode yang digunakan:
1. STFT (*Short-Time Fourier Transform*)
STFT digunakan untuk menghasilkan spectrogram, yang merepresentasikan perubahan frekuensi terhadap waktu.
- Memberikan gambaran struktur temporal–frekuensial suara

2. *Log-Mel Spectrogram*
Log-Mel spectrogram digunakan karena:
- Lebih sesuai dengan persepsi pendengaran manusia
- Banyak digunakan pada klasifikasi audio
Representasi ini juga dapat digunakan sebagai dasar pengembangan model berbasis CNN, meskipun pada penelitian ini fokus utama tetap pada fitur numerik.

**3.5 *Final Feature Set***

Berdasarkan proses ekstraksi fitur, fitur akhir yang digunakan untuk pemodelan machine learning adalah sebagai berikut:

| Fitur                | Jumlah | Deskripsi                    |
|----------------------|--------|------------------------------|
| MFCC                 | 13     | Karakteristik spektral suara |
| Zero Crossing Rate   | 1      | Kecepatan perubahan sinyal   |
| RMS Energy           | 1      | Intensitas suara             |
| Spectral Centroid    | 1      | Kecerahan suara              |

Total fitur per instance adalah 16 fitur. Jumlah ini dipilih agar model machine learning klasik dapat bekerja secara efisien tanpa kehilangan informasi penting dari sinyal audio.

**3.6 *Split Data***

1. Dataset asli menyediakan:
- TRAIN = 50
- TEST = 50
2. Pada TRAIN dilakukan split ulang:
- 80% Train
- 20% Validation
3. Validation set digunakan untuk:
- tuning hyperparameter
- pemilihan model terbaik
Setelah model optimal diperoleh, evaluasi akhir dilakukan menggunakan TEST set untuk memastikan performa model yang objektif.

**4.*Modeling* (Pemodelan)**

Tahap pemodelan bertujuan untuk membangun dan mengevaluasi model **machine learning** yang mampu mengklasifikasikan spesies burung berdasarkan fitur audio hasil ekstraksi. Model yang digunakan merupakan algoritma **machine learning** klasik yang sesuai untuk dataset berukuran kecil dengan fitur numerik hasil *feature extraction*.

**4.1 Model yang digunakan**

1. Model Utama = Random Forest
Random Forest dipilih sebagai model utama karena memiliki performa yang stabil pada dataset berukuran kecil dan menengah, serta mampu menangani data dengan karakteristik non-linear. Alasan pemilihan:
- sangat stabil untuk dataset kecil (100 sampel)
- *robust* terhadap *noisy features*
- tidak sensitif terhadap *feature scaling*
- mampu menangkap hubungan non-linear antar fitur
- *training* cepat

Random Forest bekerja dengan membangun beberapa *decision tree* dan menggabungkan hasil prediksinya melalui mekanisme *ensemble*, sehingga mengurangi risiko *overfitting*.

2. Model Alternatif = XGBoost
XGBoost digunakan sebagai model alternatif untuk dibandingkan dengan Random Forest. Alasan pemilihan:
- performa tinggi pada dataset kecil–menengah
- lebih kuat menangani struktur non-linear
- dapat outperform Random Forest pada banyak kasus audio tabular

Model ini digunakan untuk mengevaluasi apakah pendekatan boosting mampu memberikan performa lebih baik dibandingkan pendekatan bagging pada Random Forest.

3. Baseline model: KNN
KNN digunakan sebagai baseline classifier. Alasan pemilihan:
- sebagai pembanding sederhana
- Bagus sebagai minimum viable classifier

4.2 Pipeline Model :
    Raw Audio → Preprocessing → Feature Extraction → ML Model → Prediksi Kelas

- Raw Audio : Sinyal audio mentah berdimensi panjang (236.784 titik waktu).
- Preprocessing : Meliputi normalisasi sinyal dan downsampling opsional untuk efisiensi komputasi.
- Feature Extraction :
Ekstraksi 16 fitur audio:
    - MFCC (13)
    - Zero Crossing Rate (1)
    - RMS Energy (1)
    - Spectral Centroid (1)
- Model *Machine Learning* : Fitur numerik digunakan sebagai input ke model Random Forest, XGBoost, dan KNN.
- Prediksi Kelas : Model menghasilkan prediksi salah satu dari lima kelas spesies burung.
Pipeline ini dipilih karena dataset **DucksAndGeese** berupa *raw waveform* panjang yang tidak efisien digunakan langsung oleh algoritma *machine learning* klasik.

**4.3 Output Model**

- prediksi salah satu dari 5 kelas spesies burung (0–4 nama spesies)
- *confidence probability* untuk UI Streamlit (soft voting atau predict_proba)
- Untuk Streamlit:
    - waveform plot
    - *spectrogram plot*
    - hasil prediksi utama
    - *confidence chart*

**5. *Evaluation* (Evaluasi)**

Tahap evaluasi bertujuan untuk mengukur kinerja model machine learning dalam mengklasifikasikan spesies burung berdasarkan sinyal audio. Evaluasi dilakukan secara objektif menggunakan test set resmi yang disediakan oleh dataset DucksAndGeese agar hasil yang diperoleh dapat mencerminkan kemampuan generalisasi model.

Evaluasi akhir hanya dilakukan pada test set untuk menghindari data leakage.

**5.1 Metrik Evaluasi**

1. ***Accuracy*** ≥ 85% (Mengukur proporsi prediksi yang benar terhadap seluruh data uji)
2. ***Classification Report*** ;
- *precision* menunjukkan ketepatan prediksi model
- *recall* menunjukkan kemampuan model mengenali kelas yang benar
- f1-score per kelas ≥ 80% digunakan sebagai metrik utama karena menyeimbangkan precision dan recall
3. ***Confusion Matrix*** untuk :
    - Melihat distribusi prediksi benar dan salah
    - Mengidentifikasi pasangan kelas yang sering tertukar
    - Menganalisis pola kesalahan klasifikasi antar spesies

**5.2 Prosedur Evaluasi Model**

Proses evaluasi dilakukan secara bertahap sebagai berikut:
1. Pelatihan Model
Dataset TRAIN dibagi ulang menjadi:
- 80% data latih (*training set*)
- 20% data validasi (*validation set*)
2. Validasi
Data validasi digunakan untuk:
- Hyperparameter tuning Random Forest (jumlah pohon, kedalaman pohon)
- Hyperparameter tuning XGBoost (*learning rate, max depth*)
- Pemilihan model terbaik berdasarkan performa validasi
3. Pengujian Akhir
- Model terbaik diuji menggunakan test set resmi (50 sampel)
- Test set tidak digunakan dalam proses pelatihan maupun tuning
- Hasil evaluasi pada tahap ini digunakan sebagai hasil akhir penelitian

Pendekatan ini memastikan bahwa evaluasi model dilakukan secara adil dan bebas dari kebocoran data (data leakage).

**5.3 Insight evaluasi yang ditulis**

Berdasarkan hasil evaluasi, beberapa analisis yang dilakukan meliputi:
1. Perbedaan Kelas Duck vs Goose
- Analisis apakah model lebih mudah membedakan kelompok bebek (duck) dan angsa (goose)
- Umumnya, suara bebek (whistling) memiliki karakteristik frekuensi yang lebih tinggi dibandingkan suara angsa

2. Analisis Kesalahan Klasifikasi
- Identifikasi kelas yang paling sering tertukar berdasarkan confusion matrix
- Misalnya antar spesies angsa yang memiliki pola frekuensi dan durasi suara yang mirip

3. Faktor Penyebab Misclassification
- Overlap frekuensi antar spesies
- Noise lingkungan pada rekaman audio
- Variasi kualitas rekaman (kategori A dan B)

Insight ini digunakan untuk:
- Memahami keterbatasan model
- Menjadi dasar pengembangan model lebih lanjut
- Memberikan justifikasi ilmiah terhadap hasil klasifikasi yang diperoleh

**6.*Deployment* (Streamlit)**

Tahap deployment bertujuan untuk mengimplementasikan model klasifikasi yang telah dilatih ke dalam sebuah aplikasi interaktif berbasis web menggunakan Streamlit, sehingga pengguna dapat melakukan pengujian audio secara langsung tanpa harus memahami proses teknis *machine learning* di belakangnya.

**6.1 Arsitektur Deployment**

Aplikasi Streamlit berfungsi sebagai antarmuka (front-end) yang mengintegrasikan:
- proses preprocessing audio,
- ekstraksi fitur,
- inferensi model,
- serta visualisasi hasil prediksi.

Seluruh *pipeline preprocessing* dan ekstraksi fitur dibuat konsisten dengan tahap training, sehingga tidak terjadi *data leakage* maupun perbedaan distribusi fitur.

**6.2 Fitur dan Alur Kerja Aplikasi**

1. Input Audio
- upload audio .wav
- Sampling rate akan diperiksa dan disesuaikan (*downsampling* ke 16 kHz jika diperlukan)
- Durasi audio akan diseragamkan (*truncation* atau *padding*)

Tahapan preprocessing otomatis yang dilakukan sistem:
- Normalisasi sinyal audio
- *Downsampling* (jika diperlukan)
- Ekstraksi fitur:
    - MFCC
    - Zero Crossing Rate (ZCR)
    - Spectral Centroid
    - RMS Energy
    - Fitur spektral lainnya sesuai model

2. Proses Inferensi Model
- Model yang telah dilatih (Random Forest / XGBoost) dimuat dalam format .pkl
- Fitur hasil ekstraksi dikonversi ke format numerik sesuai input model
- Model menghasilkan:
    - kelas prediksi (Duck atau Goose)
    - probabilitas prediksi (confidence score)

3. Output dan Visualisasi
Aplikasi menampilkan:
- Waveform audio untuk representasi sinyal waktu
- Visualisasi MFCC (opsional) untuk representasi domain frekuensi
- Hasil prediksi spesies
- Confidence score sebagai ukuran tingkat keyakinan model
- Penjelasan singkat hasil prediksi, seperti:
    - kecenderungan frekuensi
    - intensitas energi suara

**6.3 Struktur Navigasi Aplikasi Streamlit**

Aplikasi dibagi menjadi beberapa halaman utama:
1. Home
- Deskripsi singkat proyek
- Tujuan klasifikasi suara Duck vs Goose
- Ringkasan metode yang digunakan

2. EDA (Exploratory Data Analysis)
- Visualisasi waveform audio
- Statistik dasar sinyal audio
- Contoh perbandingan suara Duck dan Goose

3. Model / Prediction
- Upload audio
- Proses preprocessing dan inferensi
- Tampilan hasil prediksi dan confidence score

4. About Dataset
- Penjelasan dataset DucksAndGeese
- Jumlah data train dan test
- Karakteristik umum sinyal audio

**6.4 Struktur File Deployment**

    /streamlit_app.py #file utama aplikasi Streamlit dan UI
    /model.pkl #model hasil training yang telah diserialisasi
    /preprocessing.py #modul normalisasi, downsampling, dan trimming audio
    /feature_extraction.py #modul ekstraksi fitur audio (MFCC, ZCR, spectral features)

**6.5 Manfaat Deployment**

- Mempermudah validasi model secara praktis
- Menjembatani hasil penelitian dengan aplikasi nyata
- Memungkinkan pengujian audio baru di luar dataset
- Menjadi bukti implementasi end-to-end pipeline machine learning